In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Stier
SRC_INPUT  = Path('/kaggle/input/datasets/adriannordbo/deep-learning-hand-gesture/src')
FREIHAND   = Path('/kaggle/input/datasets/adriannordbo/freihand-dataset')
WORK_DIR   = Path('/kaggle/working')
WORK_SRC   = WORK_DIR / 'src'
MODELL_DIR = WORK_DIR / 'modell'

# Kopier src til working (input er read-only)
if WORK_SRC.exists():
    shutil.rmtree(WORK_SRC)
shutil.copytree(SRC_INPUT, WORK_SRC)
MODELL_DIR.mkdir(exist_ok=True)

# Patch DATA_ROOT i dataset.py til å peke direkte på FreiHAND-input
dataset_file = WORK_SRC / 'dataset.py'
content = dataset_file.read_text(encoding='utf-8')
content = content.replace(
    "DATA_ROOT = PROJECT_ROOT / \"data\" / \"trene\"",
    f"DATA_ROOT = Path('{FREIHAND}')"
)
dataset_file.write_text(content, encoding='utf-8')

# Bekreft at patchen virket
rgb_dir = FREIHAND / 'training' / 'rgb'
image_count = len(list(rgb_dir.glob('*.jpg')))
print(f'FreiHAND bilder funnet: {image_count}')
assert image_count > 0, 'Ingen bilder funnet!'

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
CHECKPOINT = MODELL_DIR / 'landmark_heatmap11_best.pt'
LATEST     = MODELL_DIR / 'landmark_heatmap11_best_latest.pt'
resume_arg = f'--resume-from {LATEST}' if LATEST.exists() else ''

cmd = (
    f'python {WORK_SRC}/train.py '
    f'--data-source freihand '
    f'--epochs 30 '
    f'--batch-size 32 '
    f'--num-workers 2 '
    f'--augment '
    f'--augment-strength strong '
    f'--checkpoint {CHECKPOINT} '
    f'{resume_arg}'
)

os.chdir(WORK_SRC)
sys.path.insert(0, str(WORK_SRC))
print('Kjører:', cmd)
result = subprocess.run(cmd, shell=True, text=True)
print('Ferdig, returkode:', result.returncode)

In [ ]:
print('Output-filer:')
for f in sorted(MODELL_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')